# Tiny Agent with Tools 🛠️

**Framework:** [smolagents](https://github.com/huggingface/smolagents) (Hugging Face)

**Cost:** $0 — no API keys, no paid endpoints. Everything below runs on the free Colab CPU.

**This single notebook is both the exercise and the answer key.** Every TODO comment in the code explains *what* to fill in and *why* — the code right after it is the worked solution, already filled in and already run once so you can see the expected output before you re-run it yourself.

### What you'll build
1. A tiny knowledge base (a Python list of text snippets).
2. Two tools: `KBLookupTool` (keyword search over the KB) and `MathTool` (add/multiply).
3. A `ToolCallingAgent` that picks the right tool for each question.
4. A quick test of 3 questions, checking that tool calls + citations show up correctly.

### About the "model" in this notebook
`ToolCallingAgent` needs a language model that can read a question and decide which tool to call.
Real local LLMs (even tiny ones) need a GPU-friendly download and can still be flaky at tool-calling
if they're very small. So this notebook uses a **rule-based stand-in model** by default —
a tiny bit of Python that reads your question and picks a tool, using the exact same `Model`
interface smolagents expects. That keeps the exercise 100% free, fast, and reliable.

At the end there's an **optional bonus section** showing how to swap in a real Hugging Face model
with `TransformersModel` once you're ready — including why `sshleifer/tiny-gpt2` specifically
will *not* work well for this (it's a randomly-initialized model made for pipeline testing, not
real generation), and which small models actually do work.

**No API key is required anywhere in this notebook.**

In [ ]:
!pip install -q smolagents[transformers] wikipedia

## 1) Define the knowledge base

In [2]:
# TODO: feel free to add 1-3 more snippets of your own (5-8 total is plenty).
# Each entry needs a short 'source' tag (used for citations like [kb:agentic])
# and a one-sentence 'text'.
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))

KB entries: 5


## 2) Define the tools

In [3]:
from smolagents import Tool, ToolCallingAgent

class KBLookupTool(Tool):
    # TODO: give the tool a name and description the agent will see in its prompt.
    name = "kb_lookup_tool"
    description = (
        "Looks up relevant snippets from a small knowledge base by keyword. "
        "Use this for conceptual or 'what is X' questions."
    )
    inputs = {
        "query": {"type": "string", "description": "Keywords or a question to search the knowledge base for."}
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    # words too common to be useful as search keywords
    STOPWORDS = {"a", "an", "the", "is", "are", "of", "to", "in", "on", "and", "what", "how", "why", "do", "does"}

    def forward(self, query: str) -> str:
        # TODO: this keeps only meaningful keywords (length > 2, not a stopword).
        # Try removing the stopword filter and see how much noisier the matches get!
        keywords = {w.strip("?.,!") for w in query.lower().split() if len(w) > 2 and w not in self.STOPWORDS}
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in keywords)
        ]
        return " | ".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    # TODO: fill in the input schema. Each input needs a 'type' and 'description'.
    # 'op' is optional (nullable) since it defaults to "add" in forward().
    inputs = {
        "a": {"type": "number", "description": "The first number."},
        "b": {"type": "number", "description": "The second number."},
        "op": {"type": "string", "description": "The operation to perform: 'add' or 'multiply'.", "nullable": True},
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()
print("Tools ready:", kb_tool.name, "&", math_tool.name)

Tools ready: kb_lookup_tool & math_tool


## 3) Model — a free, reliable rule-based stand-in

In [4]:
import re
import uuid
from smolagents.models import Model, ChatMessage, ChatMessageToolCall, ChatMessageToolCallFunction, MessageRole


def _tool_call_message(name, arguments):
    """Build the ChatMessage smolagents expects when a model decides to call a tool."""
    return ChatMessage(
        role=MessageRole.ASSISTANT,
        content="",
        tool_calls=[
            ChatMessageToolCall(
                id=str(uuid.uuid4()),
                type="function",
                # TODO: pass `name` and `arguments` through to ChatMessageToolCallFunction
                function=ChatMessageToolCallFunction(name=name, arguments=arguments),
            )
        ],
    )


class RuleBasedToolCallModel(Model):
    """A tiny, deterministic stand-in for a real LLM: no download, no GPU, no API key.

    It looks at the latest user question, picks a tool with simple keyword rules,
    and once it has an observation (a tool result), turns it into a short final answer.
    This is what `ToolCallingAgent` calls on every step -- see how `generate()` matches
    the `Model` interface from smolagents.
    """

    def generate(self, messages, stop_sequences=None, response_format=None, tools_to_call_from=None, **kwargs):
        last_user_text = ""
        last_observation = None
        for m in messages:
            role = str(m.role).lower()
            content = m.content
            if isinstance(content, list):
                content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))
            if "user" in role and "New task" in (content or ""):
                last_user_text = content
            if "tool_response" in role or "tool-response" in role:
                last_observation = content

        # Step 2: we already called a tool -> turn the observation into a final answer
        if last_observation is not None:
            obs = last_observation.strip()
            if obs.lower().startswith("observation:"):
                obs = obs.split(":", 1)[1].strip()

            if obs == "No KB match." or not obs:
                answer = (
                    "I couldn't find evidence for that in the knowledge base. "
                    "Could you rephrase, or ask about agentic loops, tools, or citations?"
                )
            elif obs.startswith("["):
                # KB result looks like "[kb:agentic] Agentic AI loops plan..."
                tag, _, text = obs.partition("]")
                answer = f"{text.strip().rstrip('.')} {tag}]."
            else:
                answer = f"The result is {obs}."

            return _tool_call_message("final_answer", {"answer": answer})

        # TODO: Step 1 -- decide which tool to call based on the question text.
        # Hint: look for arithmetic keywords ('add', 'multiply', ...) AND at least
        # two numbers in the question; otherwise fall back to the KB lookup tool.
        text_lower = last_user_text.lower()
        nums = [float(n) for n in re.findall(r"-?\d+\.?\d*", text_lower)]
        wants_math = any(w in text_lower for w in ["add", "sum", "plus", "multiply", "times", "product"]) and len(nums) >= 2

        if wants_math:
            op = "multiply" if any(w in text_lower for w in ["multiply", "times", "product"]) else "add"
            return _tool_call_message("math_tool", {"a": nums[0], "b": nums[1], "op": op})

        return _tool_call_message("kb_lookup_tool", {"query": last_user_text})


model = RuleBasedToolCallModel()
print("Model ready:", model.__class__.__name__)

Model ready: RuleBasedToolCallModel


## 4) Agent

In [5]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        # TODO: write instructions telling the agent when to use each tool,
        # to keep answers short, and to cite the KB.
        "You are a small agentic assistant. Use math_tool for arithmetic questions "
        "and kb_lookup_tool for conceptual questions. Keep final answers to 2-4 sentences "
        "and include a source tag like [kb:agentic] when you used the knowledge base. "
        "If there is no evidence, say so and propose a follow-up question."
    ),
)

print(agent)

## 5) Test queries

In [6]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    # TODO: run the agent on the question q and store it in `result`
    result = agent.run(q)
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ RuleBasedToolCallModel - None ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': 12.0, 'b': 30.0, 'op': 'add'}                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 42.0

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Agentic AI loops plan, choose tools, and reflect       │
│ before answering [kb:agentic].'}                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Agentic AI loops plan, choose tools, and reflect before answering |kb:agentic].

Final answer: Agentic AI loops plan, choose tools, and reflect before answering [kb:agentic].

[Step 2: Duration 0.01 seconds]

Answer: Agentic AI loops plan, choose tools, and reflect before answering [kb:agentic].


## 6) Optional bonus: plugging in a *real* local LLM

Everything above uses a rule-based stand-in so the notebook is 100% reliable without a download.
If you want to try a genuine small language model instead, here's how -- and an important caveat.

**Why not `sshleifer/tiny-gpt2`?** That model has tiny, *randomly initialized* weights. It was built
for testing ML pipelines (making sure code runs, not that outputs make sense) -- it was never trained
on real text, so it cannot reliably follow instructions or produce valid tool-call JSON. Swapping it in
for `RuleBasedToolCallModel` above will run without errors, but the agent's tool choices will be
essentially random.

For a local model that can *actually* do tool calling, use a small **instruction-tuned** model instead,
for example `Qwen/Qwen2.5-0.5B-Instruct` (~1GB download, still free, still no API key, CPU-runnable but slow).


In [7]:
# Optional: swap in a real (but small) instruction-tuned model.
# This downloads ~1GB of weights the first time it runs and will be much slower than
# the rule-based model above -- try it once you're comfortable with the rest of the notebook.

# from smolagents import TransformersModel
#
# real_model = TransformersModel(
#     model_id="Qwen/Qwen2.5-0.5B-Instruct",
#     device_map="auto",
#     max_new_tokens=512,
# )
#
# real_agent = ToolCallingAgent(
#     tools=[kb_tool, math_tool],
#     model=real_model,
#     max_steps=4,
#     instructions=agent.instructions,
# )
#
# print(real_agent.run("Add 12 and 30."))